<a href="https://colab.research.google.com/github/LauraMazzagufo/applicazioniPython/blob/main/estrai_testo_TEI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📜 Estrazione testo da file XML/TEI → un unico XML — Batch

Estrae titolo e testo dal tag `<text><body>` di più file TEI e li raccoglie in un **unico file XML**.

**Input:** uno o più file `.xml` TEI  
**Output:** un singolo file `raccolta.xml` con questa struttura:

```xml
<raccolta>
  <poesia file="LS36_Antenati.xml">
    <titolo>Antenati</titolo>
    <testo>Stupefatto del mondo mi giunse un'età
che tiravo gran pugni nell'aria...</testo>
  </poesia>
  <poesia file="LS37_altro.xml">
    ...
  </poesia>
</raccolta>
```

## 1. Installazione dipendenze

In [19]:
!pip install lxml -q

## 2. Caricamento dei file XML

In [20]:
from google.colab import files
import os

print("Seleziona uno o più file XML/TEI da caricare...")
uploaded = files.upload()

os.makedirs("xml_input", exist_ok=True)
for filename, content in uploaded.items():
    path = os.path.join("xml_input", filename)
    with open(path, "wb") as f:
        f.write(content)
    print(f"  ✓ Caricato: {filename}")

print(f"\nTotale file caricati: {len(uploaded)}")

Seleziona uno o più file XML/TEI da caricare...


Saving LS36_Poggio_reale.xml to LS36_Poggio_reale.xml
Saving LS36_Proprietari.xml to LS36_Proprietari.xml
Saving LS36_Ritratto_d_autore.xml to LS36_Ritratto_d_autore.xml
Saving LS36_Rivolta.xml to LS36_Rivolta.xml
Saving LS36_Terre_bruciate.xml to LS36_Terre_bruciate.xml
Saving LS36_Tradimento.xml to LS36_Tradimento.xml
Saving LS36_Tradimento_FE5I.9.xml to LS36_Tradimento_FE5I.9.xml
Saving LS36_Ulisse.xml to LS36_Ulisse.xml
Saving LS36_Una_generazione.xml to LS36_Una_generazione.xml
Saving LS36_Una_stagione.xml to LS36_Una_stagione.xml
  ✓ Caricato: LS36_Poggio_reale.xml
  ✓ Caricato: LS36_Proprietari.xml
  ✓ Caricato: LS36_Ritratto_d_autore.xml
  ✓ Caricato: LS36_Rivolta.xml
  ✓ Caricato: LS36_Terre_bruciate.xml
  ✓ Caricato: LS36_Tradimento.xml
  ✓ Caricato: LS36_Tradimento_FE5I.9.xml
  ✓ Caricato: LS36_Ulisse.xml
  ✓ Caricato: LS36_Una_generazione.xml
  ✓ Caricato: LS36_Una_stagione.xml

Totale file caricati: 10


## 3. Funzioni di estrazione

In [21]:
from lxml import etree
from pathlib import Path

NS = "http://www.tei-c.org/ns/1.0"
ns = {"tei": NS}


def testo_nodo(nodo):
    """Estrae tutto il testo da un nodo ricorsivamente, con pulizia spazi."""
    return " ".join("".join(nodo.itertext()).split())


def estrai_poesia(filepath):
    """
    Legge un file TEI e restituisce un elemento <poesia> con titolo e testo.
    Restituisce None in caso di errore.
    """
    try:
        tree = etree.parse(str(filepath))
        root = tree.getroot()
    except etree.XMLSyntaxError as e:
        print(f"  ✗ Errore XML in {filepath.name}: {e}")
        return None

    text_node = root.find("tei:text", ns)
    if text_node is None:
        print(f"  ✗ Nessun <text> trovato in {filepath.name}")
        return None

    body = text_node.find("tei:body", ns)
    if body is None:
        print(f"  ✗ Nessun <body> trovato in {filepath.name}")
        return None

    poesia = etree.Element("poesia")
    poesia.set("file", filepath.name)

    # Titolo
    head = body.find("tei:head", ns)
    titolo_el = etree.SubElement(poesia, "titolo")
    titolo_el.text = testo_nodo(head).strip() if head is not None else ""

    # Testo unico: versi separati da \n, strofe da \n\n
    strofe = body.findall(".//tei:lg", ns)
    blocchi = [
        "\n".join(testo_nodo(l) for l in lg.findall("tei:l", ns))
        for lg in strofe
    ]
    testo_el = etree.SubElement(poesia, "testo")
    testo_el.text = "\n\n".join(blocchi)

    return poesia


print("✓ Funzioni caricate.")

✓ Funzioni caricate.


## 4. Elaborazione batch → unico file XML

In [22]:
# ── OPZIONI ──────────────────────────────────────────────
INPUT_DIR   = "xml_input"    # cartella con i file TEI in input
OUTPUT_FILE = "raccolta.xml" # nome del file XML di output
# ─────────────────────────────────────────────────────────

from pathlib import Path

file_xml = sorted(Path(INPUT_DIR).glob("*.xml"))
print(f"File XML trovati: {len(file_xml)}\n")

# Radice del documento di output
raccolta = etree.Element("raccolta")

ok, errori = 0, 0

for xml_file in file_xml:
    print(f"⏳ {xml_file.name} ...", end=" ")
    poesia = estrai_poesia(xml_file)
    if poesia is None:
        errori += 1
        continue
    raccolta.append(poesia)
    print("✓")
    ok += 1

# Scrittura file unico
etree.ElementTree(raccolta).write(
    OUTPUT_FILE,
    encoding="utf-8",
    xml_declaration=True,
    pretty_print=True
)

print(f"\n{'='*40}")
print(f"  Elaborati con successo : {ok}")
print(f"  Errori                 : {errori}")
print(f"  File di output         : {OUTPUT_FILE}")
print(f"{'='*40}")

File XML trovati: 40

⏳ LS36_Disciplina.xml ... ✓
⏳ LS36_Disciplina_antica.xml ... ✓
⏳ LS36_Donne_appassionate.xml ... ✓
⏳ LS36_Due_sigarette.xml ... ✓
⏳ LS36_Esterno.xml ... ✓
⏳ LS36_Gente_che_non_capisce.xml ... ✓
⏳ LS36_Gente_spaesata.xml ... ✓
⏳ LS36_Grappa_a_settembre.xml ... ✓
⏳ LS36_I_mari_del_sud.xml ... ✓
⏳ LS36_Il_dio-caprone.xml ... ✓
⏳ LS36_Il_tempo_passa.xml ... ✓
⏳ LS36_Indisciplina.xml ... ✓
⏳ LS36_La_cena_triste.xml ... ✓
⏳ LS36_Lavorare_stanca(II).xml ... ✓
⏳ LS36_LegnaVerde.xml ... ✓
⏳ LS36_Luna_d_agosto.xml ... ✓
⏳ LS36_Mania_di_solitudine.xml ... ✓
⏳ LS36_Maternita.xml ... ✓
⏳ LS36_Mediterranea.xml ... ✓
⏳ LS36_Ozio.xml ... ✓
⏳ LS36_Paesaggio_I.xml ... ✓
⏳ LS36_Paesaggio_II.xml ... ✓
⏳ LS36_Paesaggio_III.xml ... ✓
⏳ LS36_Paesaggio_IV.xml ... ✓
⏳ LS36_Paesaggio_V.xml ... ✓
⏳ LS36_Paesaggio_VI.xml ... ✓
⏳ LS36_Paternita.xml ... ✓
⏳ LS36_Pensieri_di_Deola.xml ... ✓
⏳ LS36_Pensieri_di_Dina.xml ... ✓
⏳ LS36_Piaceri_notturni.xml ... ✓
⏳ LS36_Poggio_reale.xml ... ✓
⏳ LS36_

## 5. Anteprima del file di output

In [23]:
print(Path(OUTPUT_FILE).read_text(encoding="utf-8"))

<?xml version='1.0' encoding='UTF-8'?>
<raccolta>
  <poesia file="LS36_Disciplina.xml">
    <titolo>Disciplina</titolo>
    <testo>I lavori cominciano all'alba. Ma noi cominciamo
un po' prima dell'alba a incontrare noi stessi
nella gente che va per la strada. Ciascuno ricorda
di esser solo e aver sonno, scoprendo i passanti
radi - ognuno trasogna fra sé,
tanto sa che nell'alba spalancherà gli occhi.

Quando viene il mattino ci trova stupiti
a fissare il lavoro che adesso comincia.
Ma non siamo più soli e nessuno ha più sonno
e pensiamo con calma i pensieri del giorno
fino a dare in sorrisi. Nel sole che torna
siamo tutti convinti. Ma a volte un pensiero
meno chiaro - un sogghigno - ci coglie improvviso
e torniamo a guardare come prima del sole.
La città chiara assiste ai lavori e ai sogghigni.
Nulla può disturbare il mattino. Ogni cosa
può accadere e ci basta alzare la testa
dal lavoro e guardare. Ragazzi scappati
che non fanno ancor nulla, camminano in strada
e qualcuno anche corre. L

## 6. Download

In [24]:
from google.colab import files
files.download(OUTPUT_FILE)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>